# Load Packages

# Assing basin id


In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams as rc
import matplotlib
import math
import matplotlib.colors as mcolors
import os
from pyproj import Proj, transform
import linear_operator
import lasio
from pyproj import Transformer
import geopandas as gpd
from shapely.geometry import Polygon


configs = {
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'font.sans-serif': ['Times New Roman'],
     #'font.sans-serif': ['Microsoft YaHei', 'SimHei', 'Arial', 'Liberation Sans', 'sans-serif'],
    #'font.sans-serif': ['Palatino Linotype', 'Microsoft YaHei', 'SimHei', 'Arial', 'Liberation Sans', 'sans-serif'],
    'axes.grid': True,
    'grid.alpha': 0.5,
    'grid.linestyle': '--',
    'mathtext.fontset': "cm",
    # 'mathtext.rm': "asana math",
    # 'mathtext.cal': "asana math",
    "font.size": 14,
    # 'axes.labelsize': 'large',
    # 'axes.labelpad': 8.0,
    # 'xtick.major.pad': 9.0,
}
rc.update(configs)

In [3]:
cmap = matplotlib.colormaps.get_cmap("coolwarm")
red_color = cmap(0.9)
blue_color = cmap(0.1)
gray_color = cmap(0.5)

color_maps = {
    "input": gray_color,
    "Expected": red_color,
    "Predicted": blue_color,
}

dir_fig = '../Data/Figure/'
dir_out = '../Data/Vel_data/'

# Borehole

In [4]:
#read the entire data
df_measure = pd.read_csv(dir_out+'Preprocessing_borehole/vs_data_output.txt', encoding='utf-8')
df_measure = df_measure.rename(columns={'metadata_id': 'velocity_metadata_id', 'z_2.5': 'Z_2.5',
                                       'vs_middle_depth': 'vs_depth'})
df_measure

,velocity_metadata_id,vs_depth,vs_value,vs_top_depth,vs_bottom_depth,test_id,velocity_name,velocity_method,velocity_date,velocity_quality,...,slope_resolution,terrain_class,site_easting,site_northing,test_easting,test_northing,vs_model,surf_elevation,Z_2.5,diff
0,610,1.00,155.0,0.5,1.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,30 arc-sec,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,938.187,251.052,2380.0,-1.800524
1,610,2.00,170.2,1.5,2.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,30 arc-sec,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,939.068,251.052,2380.0,-1.707914
2,610,3.00,196.5,2.5,3.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,30 arc-sec,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,939.949,251.052,2380.0,-1.565163
3,610,4.00,191.9,3.5,4.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,30 arc-sec,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,940.830,251.052,2380.0,-1.589788
4,610,5.00,202.0,4.5,5.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,30 arc-sec,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,941.711,251.052,2380.0,-1.539431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11207,2479,12.60,354.0,9.3,15.9,2419,VS,MASW,NaN,NaN,...,30 arc-sec,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,894.882,54.938,3620.0,-0.927395
11208,2479,21.15,387.0,15.9,26.4,2419,VS,MASW,NaN,NaN,...,30 arc-sec,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,899.010,54.938,3620.0,-0.842869
11209,2479,34.80,445.0,26.4,43.2,2419,VS,MASW,NaN,NaN,...,30 arc-sec,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,905.600,54.938,3620.0,-0.710523
11210,2479,56.60,540.0,43.2,70.0,2419,VS,MASW,NaN,NaN,...,30 arc-sec,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,916.125,54.938,3620.0,-0.528584


In [6]:
gdf = gpd.read_file('../Data/Raw_data/GIS/CVM-S4_basins_boundary.gpkg')
gdf_utm = gdf.to_crs("EPSG:32611")
line = gdf_utm.geometry.iloc[0]
coords = list(line.coords)
poly = Polygon(coords)

pts = gpd.GeoDataFrame(
    df_measure.copy(),
    geometry=gpd.points_from_xy(df_measure["site_easting"], df_measure["site_northing"]),
    crs="EPSG:32611")

mask = pts.geometry.within(poly) | pts.geometry.touches(poly)
df_measure = pts.loc[mask].drop(columns="geometry").reset_index(drop=True)
df_measure

,velocity_metadata_id,vs_depth,vs_value,vs_top_depth,vs_bottom_depth,test_id,velocity_name,velocity_method,velocity_date,velocity_quality,...,slope_resolution,terrain_class,site_easting,site_northing,test_easting,test_northing,vs_model,surf_elevation,Z_2.5,diff
0,610,1.00,155.0,0.5,1.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,30 arc-sec,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,938.187,251.052,2380.0,-1.800524
1,610,2.00,170.2,1.5,2.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,30 arc-sec,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,939.068,251.052,2380.0,-1.707914
2,610,3.00,196.5,2.5,3.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,30 arc-sec,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,939.949,251.052,2380.0,-1.565163
3,610,4.00,191.9,3.5,4.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,30 arc-sec,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,940.830,251.052,2380.0,-1.589788
4,610,5.00,202.0,4.5,5.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,30 arc-sec,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,941.711,251.052,2380.0,-1.539431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11026,2479,12.60,354.0,9.3,15.9,2419,VS,MASW,NaN,NaN,...,30 arc-sec,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,894.882,54.938,3620.0,-0.927395
11027,2479,21.15,387.0,15.9,26.4,2419,VS,MASW,NaN,NaN,...,30 arc-sec,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,899.010,54.938,3620.0,-0.842869
11028,2479,34.80,445.0,26.4,43.2,2419,VS,MASW,NaN,NaN,...,30 arc-sec,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,905.600,54.938,3620.0,-0.710523
11029,2479,56.60,540.0,43.2,70.0,2419,VS,MASW,NaN,NaN,...,30 arc-sec,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,916.125,54.938,3620.0,-0.528584


In [7]:
gdf_basins = gpd.read_file("../Data/Raw_data/Basins/LA_Basins_merged3.shp")

gdf_basins_utm = gdf_basins.to_crs(epsg=32611)


gdf_pts = gpd.GeoDataFrame(
    df_measure.copy(),
    geometry=gpd.points_from_xy(df_measure["site_easting"], df_measure["site_northing"]),
    crs="EPSG:32611"  
)

basin_cols = ["OBJECTID", "Basin_Numb", "Basin_Name", "geometry"]
gdf_basin_small = gdf_basins_utm[basin_cols].copy()


joined = gpd.sjoin(
    gdf_pts,
    gdf_basin_small,
    how="left",
    predicate="within" 
)
joined

,velocity_metadata_id,vs_depth,vs_value,vs_top_depth,vs_bottom_depth,test_id,velocity_name,velocity_method,velocity_date,velocity_quality,...,test_northing,vs_model,surf_elevation,Z_2.5,diff,geometry,index_right,OBJECTID,Basin_Numb,Basin_Name
0,610,1.00,155.0,0.5,1.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,3.786794e+06,938.187,251.052,2380.0,-1.800524,POINT (352134.047 3786793.966),1.0,2.0,4-012,SAN FERNANDO VALLEY
1,610,2.00,170.2,1.5,2.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,3.786794e+06,939.068,251.052,2380.0,-1.707914,POINT (352134.047 3786793.966),1.0,2.0,4-012,SAN FERNANDO VALLEY
2,610,3.00,196.5,2.5,3.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,3.786794e+06,939.949,251.052,2380.0,-1.565163,POINT (352134.047 3786793.966),1.0,2.0,4-012,SAN FERNANDO VALLEY
3,610,4.00,191.9,3.5,4.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,3.786794e+06,940.830,251.052,2380.0,-1.589788,POINT (352134.047 3786793.966),1.0,2.0,4-012,SAN FERNANDO VALLEY
4,610,5.00,202.0,4.5,5.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,3.786794e+06,941.711,251.052,2380.0,-1.539431,POINT (352134.047 3786793.966),1.0,2.0,4-012,SAN FERNANDO VALLEY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11026,2479,12.60,354.0,9.3,15.9,2419,VS,MASW,NaN,NaN,...,3.762835e+06,894.882,54.938,3620.0,-0.927395,POINT (400420.746 3762835.246),4.0,1.0,4-011,COASTAL PLAIN OF LOS ANGELES
11027,2479,21.15,387.0,15.9,26.4,2419,VS,MASW,NaN,NaN,...,3.762835e+06,899.010,54.938,3620.0,-0.842869,POINT (400420.746 3762835.246),4.0,1.0,4-011,COASTAL PLAIN OF LOS ANGELES
11028,2479,34.80,445.0,26.4,43.2,2419,VS,MASW,NaN,NaN,...,3.762835e+06,905.600,54.938,3620.0,-0.710523,POINT (400420.746 3762835.246),4.0,1.0,4-011,COASTAL PLAIN OF LOS ANGELES
11029,2479,56.60,540.0,43.2,70.0,2419,VS,MASW,NaN,NaN,...,3.762835e+06,916.125,54.938,3620.0,-0.528584,POINT (400420.746 3762835.246),4.0,1.0,4-011,COASTAL PLAIN OF LOS ANGELES


In [8]:
df_measure['basin_idnum'] = joined['OBJECTID']
mask_nan = df_measure["basin_idnum"].isna()
gdf_pts_nan = gdf_pts.loc[mask_nan].copy()

if len(gdf_pts_nan) > 0:
    nearest = gpd.sjoin_nearest(
        gdf_pts_nan,
        gdf_basin_small,
        how="left",
        distance_col="dist_to_basin_m"
    )

    basin_fill = np.where(
        nearest["dist_to_basin_m"] <= 1000,
        nearest["OBJECTID"],
        0)
    df_measure.loc[mask_nan, "basin_idnum"] = basin_fill

df_measure["basin_idnum"] = df_measure["basin_idnum"].astype(int)
df_measure

,velocity_metadata_id,vs_depth,vs_value,vs_top_depth,vs_bottom_depth,test_id,velocity_name,velocity_method,velocity_date,velocity_quality,...,terrain_class,site_easting,site_northing,test_easting,test_northing,vs_model,surf_elevation,Z_2.5,diff,basin_idnum
0,610,1.00,155.0,0.5,1.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,938.187,251.052,2380.0,-1.800524,2
1,610,2.00,170.2,1.5,2.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,939.068,251.052,2380.0,-1.707914,2
2,610,3.00,196.5,2.5,3.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,939.949,251.052,2380.0,-1.565163,2
3,610,4.00,191.9,3.5,4.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,940.830,251.052,2380.0,-1.589788,2
4,610,5.00,202.0,4.5,5.5,542,ELC,Suspension Logging,1996-09-26,NaN,...,4,352134.046903,3.786794e+06,352134.046903,3.786794e+06,941.711,251.052,2380.0,-1.539431,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11026,2479,12.60,354.0,9.3,15.9,2419,VS,MASW,NaN,NaN,...,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,894.882,54.938,3620.0,-0.927395,1
11027,2479,21.15,387.0,15.9,26.4,2419,VS,MASW,NaN,NaN,...,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,899.010,54.938,3620.0,-0.842869,1
11028,2479,34.80,445.0,26.4,43.2,2419,VS,MASW,NaN,NaN,...,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,905.600,54.938,3620.0,-0.710523,1
11029,2479,56.60,540.0,43.2,70.0,2419,VS,MASW,NaN,NaN,...,8,400420.745906,3.762835e+06,400420.745906,3.762835e+06,916.125,54.938,3620.0,-0.528584,1


# Log


In [9]:
df_sonic = pd.read_csv(dir_out+'Preprocessing_log/vs_data_combined.txt', encoding='utf-8')
df_sonic = df_sonic.rename(columns={'X': 'easting', 'Y': 'northing',
                                       'depth': 'vs_depth'})
df_sonic

,DEPTH_m,velocity_mps,easting,northing,source_file,TVDSS,velocity,velocity_metadata_id,longitude,latitude,vs_UCVM,vp_UCVM,surf_elevation,ratio,vs,vs_depth,Z_2.5,diff
0,43.5280,1713.535764,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,43.528000,1702.435158,-29,-118.218003,33.912748,867.819,1684.427,23.596,0.515201,877.096827,67.124000,4400.0,0.010634
1,53.5280,1756.154846,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,53.528000,1723.341882,-29,-118.218003,33.912748,871.262,1692.589,23.596,0.514751,887.092079,77.124000,4400.0,0.018006
2,63.5280,1738.138676,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,63.528000,1750.140249,-29,-118.218003,33.912748,874.706,1700.750,23.596,0.514306,900.107703,87.124000,4400.0,0.028627
3,73.5280,1699.290684,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,73.528000,1779.210020,-29,-118.218003,33.912748,878.150,1708.912,23.596,0.513865,914.273689,97.124000,4400.0,0.040313
4,83.5280,1748.027201,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,83.528000,1806.930958,-29,-118.218003,33.912748,881.593,1717.073,23.596,0.513428,927.728573,107.124000,4400.0,0.051009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3177,1956.3772,3072.301908,387298.0938,3767711.984,tuo_2,1873.377197,3072.714253,-28,-118.221002,34.044045,2107.562,3453.733,83.353,0.610227,1875.053977,1956.730197,2500.0,-0.116894
3178,1966.3772,3274.496954,387298.0938,3767711.984,tuo_2,1883.377197,3273.990138,-28,-118.221002,34.044045,2115.348,3464.896,83.353,0.610508,1998.798374,1966.730197,2500.0,-0.056673
3179,1976.3772,3338.298432,387298.0938,3767711.984,tuo_2,1893.377197,3338.257238,-28,-118.221002,34.044045,2123.133,3476.058,83.353,0.610788,2038.966008,1976.730197,2500.0,-0.040450
3180,1986.3772,3316.756804,387298.0938,3767711.984,tuo_2,1903.377197,3316.470704,-28,-118.221002,34.044045,2130.919,3487.221,83.353,0.611065,2026.579456,1986.730197,2500.0,-0.050204


In [10]:
df_sonic['basin_idnum'] = 1
df_sonic

,DEPTH_m,velocity_mps,easting,northing,source_file,TVDSS,velocity,velocity_metadata_id,longitude,latitude,vs_UCVM,vp_UCVM,surf_elevation,ratio,vs,vs_depth,Z_2.5,diff,basin_idnum
0,43.5280,1713.535764,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,43.528000,1702.435158,-29,-118.218003,33.912748,867.819,1684.427,23.596,0.515201,877.096827,67.124000,4400.0,0.010634,1
1,53.5280,1756.154846,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,53.528000,1723.341882,-29,-118.218003,33.912748,871.262,1692.589,23.596,0.514751,887.092079,77.124000,4400.0,0.018006,1
2,63.5280,1738.138676,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,63.528000,1750.140249,-29,-118.218003,33.912748,874.706,1700.750,23.596,0.514306,900.107703,87.124000,4400.0,0.028627,1
3,73.5280,1699.290684,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,73.528000,1779.210020,-29,-118.218003,33.912748,878.150,1708.912,23.596,0.513865,914.273689,97.124000,4400.0,0.040313,1
4,83.5280,1748.027201,387402.0000,3753149.500,HVD_CVMH_soccom_1.las,83.528000,1806.930958,-29,-118.218003,33.912748,881.593,1717.073,23.596,0.513428,927.728573,107.124000,4400.0,0.051009,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3177,1956.3772,3072.301908,387298.0938,3767711.984,tuo_2,1873.377197,3072.714253,-28,-118.221002,34.044045,2107.562,3453.733,83.353,0.610227,1875.053977,1956.730197,2500.0,-0.116894,1
3178,1966.3772,3274.496954,387298.0938,3767711.984,tuo_2,1883.377197,3273.990138,-28,-118.221002,34.044045,2115.348,3464.896,83.353,0.610508,1998.798374,1966.730197,2500.0,-0.056673,1
3179,1976.3772,3338.298432,387298.0938,3767711.984,tuo_2,1893.377197,3338.257238,-28,-118.221002,34.044045,2123.133,3476.058,83.353,0.610788,2038.966008,1976.730197,2500.0,-0.040450,1
3180,1986.3772,3316.756804,387298.0938,3767711.984,tuo_2,1903.377197,3316.470704,-28,-118.221002,34.044045,2130.919,3487.221,83.353,0.611065,2026.579456,1986.730197,2500.0,-0.050204,1


# Save data

In [11]:
df_measure.to_csv('../Data/Vel_data/Training_data/'+'df_measure.txt', index=False,
                  encoding='utf-8',
                  lineterminator='\n')

In [12]:
df_sonic.to_csv('../Data/Vel_data/Training_data/'+'df_sonic.txt', index=False,
                  encoding='utf-8',
                  lineterminator='\n')